# Pre-Reg Checks
1. ANOVA sequential and nested comparisons
2. Cross-tab of `is_no_show` and `total_hours_paid` 

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
%cd "/Users/Sonia/Documents/SYEP"

df = pd.read_csv('data/to_use/analysis_frame_worksite.csv')

/Applications/Positron.app/Contents/Resources/app/extensions/positron-python/python_files/lib/ipykernel/py3/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/Users/sonia/Documents/SYEP


In [4]:
TARGET = "returned"
MIN_SITE_N = 20          # sites below this pool into "other"

# ---------------------------------------------------------
# 0. Restrict and collapse the long tail
# ---------------------------------------------------------
d = df[df["worksite_status"].eq("active")].copy()
d[TARGET] = d[TARGET].astype(int)
d = d.dropna(subset=[TARGET, "provider", "worksite_cluster_id", "Year"])

sizes = d["worksite_cluster_id"].value_counts()
keep = sizes.index[sizes >= MIN_SITE_N]
d["site"] = d["worksite_cluster_id"].where(
    d["worksite_cluster_id"].isin(keep), "other")

print(f"{len(d):,} rows | {d['provider'].nunique()} providers | "
      f"{d['site'].nunique()} site levels "
      f"({len(keep)} kept at n>={MIN_SITE_N}, rest pooled)")

for c in ["Year", "provider", "site"]:
    d[c] = d[c].astype("category")

103,473 rows | 46 providers | 1124 site levels (1123 kept at n>=20, rest pooled)


In [5]:
# ---------------------------------------------------------
# 1. Sequential (Type I) decomposition
# ---------------------------------------------------------
m_full = smf.ols(f"{TARGET} ~ C(Year) + C(provider) + C(site)", data=d).fit()
aov = sm.stats.anova_lm(m_full, typ=1)

aov["pct_of_total_var"] = 100 * aov["sum_sq"] / aov["sum_sq"].sum()
print("\n--- Sequential ANOVA (share of total variance) ---")
print(aov[["df", "sum_sq", "pct_of_total_var"]].round(4))


--- Sequential ANOVA (share of total variance) ---
                   df      sum_sq  pct_of_total_var
C(Year)           2.0     50.2365            0.2085
C(provider)      45.0    492.9830            2.0460
C(site)        1123.0   1167.7232            4.8464
Residual     102302.0  22383.8237           92.8991


In [7]:
# ---------------------------------------------------------
# 2. Nested comparisons, both directions
# ---------------------------------------------------------
m_year = smf.ols(f"{TARGET} ~ C(Year)", data=d).fit()
m_prov = smf.ols(f"{TARGET} ~ C(Year) + C(provider)", data=d).fit()
m_site = smf.ols(f"{TARGET} ~ C(Year) + C(site)", data=d).fit()

fits = pd.DataFrame({
    "model": ["Year", "Year+Provider", "Year+Site", "Year+Provider+Site"],
    "params": [m.df_model for m in (m_year, m_prov, m_site, m_full)],
    "r2": [m.rsquared for m in (m_year, m_prov, m_site, m_full)],
    "r2_adj": [m.rsquared_adj for m in (m_year, m_prov, m_site, m_full)],
})
print("\n--- Nested fits ---")
print(fits.round(4).to_string(index=False))

print("\n--- Does site add anything beyond provider? ---")
print(sm.stats.anova_lm(m_prov, m_full).round(4))

print("\n--- Does provider add anything beyond site? ---")
print(sm.stats.anova_lm(m_site, m_full).round(4))


--- Nested fits ---
             model  params     r2  r2_adj
              Year     2.0 0.0021  0.0021
     Year+Provider    47.0 0.0225  0.0221
         Year+Site  1125.0 0.0679  0.0576
Year+Provider+Site  1170.0 0.0710  0.0604

--- Does site add anything beyond provider? ---
   df_resid         ssr  df_diff    ss_diff       F  Pr(>F)
0  103425.0  23551.5470      0.0        NaN     NaN     NaN
1  102302.0  22383.8237   1123.0  1167.7232  4.7524     0.0

--- Does provider add anything beyond site? ---
   df_resid         ssr  df_diff  ss_diff       F  Pr(>F)
0  102347.0  22459.8872      0.0      NaN     NaN     NaN
1  102302.0  22383.8237     45.0  76.0635  7.7253     0.0


Most of the provider-level variation may actually reflect **which worksites providers operate**, rather than differences attributable to the providers themselves. Key word: **may**. Providers don't serve youth directly; worksites do, with providers acting as the intermediary. A provider's overall rate is therefore a weighted average of the outcomes at its worksites. Consistent with this, adding site increases $R^2$ by about **4.9 percentage points** beyond provider, while adding provider increases $R^2$ by only about **0.3 percentage points** beyond site.

In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
%cd "/Users/Sonia/Documents/SYEP"

df = pd.read_csv('data/to_use/analysis_frame_worksite.csv')

/Users/sonia/Documents/SYEP


In [ ]:
tab = pd.crosstab(df['is_no_show'], df['total_hours_paid'].eq(0), dropna=False)
print(tab)

total_hours_paid  False  True 
is_no_show                    
False             93699  11534
NaN               12756   5932
True                752   5744


In [12]:
print(pd.crosstab(df['is_no_show'], df['total_hours_paid'].eq(0),
                  normalize='index'))

total_hours_paid     False     True 
is_no_show                          
False             0.890396  0.109604
True              0.115764  0.884236


No-show flag vs. zero paid hours. The two disagree in **both directions**, asymmetrically: 752 flagged no-shows have positive hours, while 11,534 unflagged participants have zero. Since the flag identifies only about a third of youth who never worked, it reflects whether a provider typed it into the worksite field rather than whether the youth appeared. `total_hours_paid = 0` is therefore the substantive measure of non-participation; `is_no_show` is retained as a documentation indicator, not a behavioral one. Both enter the model, with the flag interpreted as a data-quality covariate.